# Day 13: Project 1 — Baseline Model (Logistic Regression)

**Dataset:** diabetes_clean.csv (from Day 12)
**Target:** `readmitted_binary` (11.2% positive)
**Goal:** Simple baseline with class_weight='balanced', CV, threshold tuning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             roc_curve, precision_recall_curve, f1_score, precision_score, recall_score)
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Load cleaned data
df = pd.read_csv('data/diabetes_clean.csv')
print(f"Loaded: {df.shape}")

Loaded: (101766, 57)


In [2]:
# 1. DEFINE FEATURES & TARGET
# Exclude: IDs, raw diag codes (use _cat versions), original target
exclude_cols = ['encounter_id', 'patient_nbr', 'diag_1', 'diag_2', 'diag_3', 
                'readmitted', 'readmitted_binary']

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"Features ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df['readmitted_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")

Features (50): ['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat', 'prior_admissions', 'med_change_count', 'los_category', 'num_diagnoses_cat', 'age_midpoint']
Train: (81412, 50), Test: (20354, 50)
Train class dist: {0: 72326, 1: 9086}


In [3]:
# 2. IDENTIFY NUMERIC vs CATEGORICAL
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric ({len(numeric_features)}): {numeric_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")

Numeric (14): ['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'prior_admissions', 'med_change_count', 'age_midpoint']
Categorical (36): ['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat', 'los_category', 'num_diagnoses_cat']


In [4]:
# 3. BUILD PIPELINE
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

print("Pipeline built")

Pipeline built


In [5]:
# 4. CROSS-VALIDATION
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print("=== 5-Fold CV Results ===")
print(f"ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Individual folds: {[f'{s:.4f}' for s in cv_scores]}")

=== 5-Fold CV Results ===
ROC-AUC: 0.6397 (+/- 0.0049)
Individual folds: ['0.6363', '0.6430', '0.6442', '0.6318', '0.6433']


In [6]:
# 5. FIT ON FULL TRAIN, EVALUATE ON TEST
pipeline.fit(X_train, y_train)

y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

test_roc = roc_auc_score(y_test, y_proba)
test_pr = average_precision_score(y_test, y_proba)

print("=== Test Set Results ===")
print(f"ROC-AUC: {test_roc:.4f}")
print(f"PR-AUC:  {test_pr:.4f}")
print(classification_report(y_test, y_pred, target_names=['No Readmit', 'Readmit <30']))

=== Test Set Results ===
ROC-AUC: 0.6518
PR-AUC:  0.2050
              precision    recall  f1-score   support

  No Readmit       0.92      0.66      0.77     18083
 Readmit <30       0.17      0.56      0.26      2271

    accuracy                           0.65     20354
   macro avg       0.55      0.61      0.52     20354
weighted avg       0.84      0.65      0.71     20354



In [7]:
# 6. THRESHOLD TUNING (Youden's J)
def optimal_threshold(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    return thresholds[best_idx], j_scores[best_idx]

opt_thresh, j_score = optimal_threshold(y_test, y_proba)
print(f"Optimal threshold: {opt_thresh:.4f} (Youden's J = {j_score:.4f})")

y_pred_opt = (y_proba >= opt_thresh).astype(int)
print(f"\n=== At Optimal Threshold ({opt_thresh:.4f}) ===")
print(f"Precision: {precision_score(y_test, y_pred_opt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_opt):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_opt):.4f}")
print(classification_report(y_test, y_pred_opt, target_names=['No Readmit', 'Readmit <30']))

Optimal threshold: 0.5103 (Youden's J = 0.2231)

=== At Optimal Threshold (0.5103) ===
Precision: 0.1780
Recall:    0.5310
F1:        0.2667
              precision    recall  f1-score   support

  No Readmit       0.92      0.69      0.79     18083
 Readmit <30       0.18      0.53      0.27      2271

    accuracy                           0.67     20354
   macro avg       0.55      0.61      0.53     20354
weighted avg       0.84      0.67      0.73     20354



In [8]:
# 7. FEATURE COEFFICIENTS (interpretability)
preprocessor.fit(X_train)
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_feature_names)

coef = pipeline.named_steps['classifier'].coef_[0]
coef_df = pd.DataFrame({'feature': all_feature_names, 'coefficient': coef})
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False)

print("=== Top 20 Features by |Coefficient| ===")
print(coef_df.head(20).to_string(index=False))

=== Top 20 Features by |Coefficient| ===
                                   feature  coefficient
medical_specialty_Pediatrics-Endocrinology    -1.655400
              medical_specialty_Gynecology    -1.492191
          medical_specialty_Otolaryngology    -1.226172
                      diag_3_cat_Pregnancy    -1.176552
    medical_specialty_AllergyandImmunology     1.110854
                      diag_2_cat_Pregnancy    -1.091640
              medical_specialty_Hematology     0.935171
                             weight_[0-25)     0.928877
                      diag_1_cat_Pregnancy     0.895769
         medical_specialty_Surgery-Plastic     0.885286
                        diag_1_cat_Missing     0.881676
 medical_specialty_Pediatrics-CriticalCare    -0.847664
      medical_specialty_InfectiousDiseases     0.842309
               medical_specialty_Radiology     0.659614
       medical_specialty_SurgicalSpecialty    -0.648720
                             payer_code_WC    -0.626012
       

In [9]:
# 8. SAVE BASELINE PIPELINE
import os
os.makedirs('../models', exist_ok=True)
model_path = 'models/readmission_lr_baseline.joblib'
joblib.dump(pipeline, model_path)

print(f"Baseline pipeline saved to {model_path}")

Baseline pipeline saved to models/readmission_lr_baseline.joblib
